<a href="https://colab.research.google.com/github/esetafinau-gif/AAI2025/blob/2026fall/Part_2_Predict_Customer_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 150

age = np.random.randint(18, 70, size=n)
monthly_usage = np.random.uniform(10, 300, size=n).round(1)
purchase_amount = np.random.uniform(20, 500, size=n).round(2)
customer_service_calls = np.random.randint(0, 8, size=n)
region = np.random.choice(["North", "South", "East", "West"], size=n)

# Higher calls and lower usage increase churn probability
logit = -2 + (customer_service_calls * 0.5) - (monthly_usage * 0.01)
prob = 1 / (1 + np.exp(-logit))
churn = (prob > np.random.uniform(0, 1, size=n)).astype(int)

churn_df = pd.DataFrame(
    {
        "age": age,
        "monthly_usage": monthly_usage,
        "purchase_amount": purchase_amount,
        "customer_service_calls": customer_service_calls,
        "region": region,
        "churn": churn,
    }
)

# Source citation comment required by assignment:
# Data Source: Generated synthetic customer churn dataset (150 records).
churn_df.to_csv("churn_data.csv", index=False)

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Data Source: Generated synthetic customer churn dataset (150+ records).
# Kaggle Reference: https://www.kaggle.com/datasets/ziya07/customer-churn-prediction-dataset

# 1. Load Dataset
df_churn = pd.read_csv("churn_data.csv")

# 2. Separate Numerical and Categorical Features
num_cols = ["age", "monthly_usage", "purchase_amount", "customer_service_calls"]
cat_cols = ["region"]

# 3. Scale Numerical Features & One-Hot Encode Categorical Features
scaler_churn = StandardScaler()
X_num_scaled = scaler_churn.fit_transform(df_churn[num_cols])

encoder_churn = OneHotEncoder(sparse_output=False)
X_cat_encoded = encoder_churn.fit_transform(df_churn[cat_cols])

# Combine processed features into single array
X_churn_features = np.hstack([X_num_scaled, X_cat_encoded])
y_churn = df_churn["churn"]

# 4. Train Logistic Regression Model
model_churn = LogisticRegression()
model_churn.fit(X_churn_features, y_churn)

# 5. Extract Feature Impact & Display Model Coefficients
# Positive log-odds increase churn risk; negative log-odds decrease churn risk.
encoded_region_names = encoder_churn.get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(encoded_region_names)

print("--- Part 2: Model Coefficients & Feature Impact ---")
for name, coef in zip(all_feature_names, model_churn.coef_[0]):
  print(f"{name}: {coef:.4f}")
print(f"Intercept: {model_churn.intercept_[0]:.4f}")

# 6. Predict Churn Risk for Sample Customer
# Inputs: age=35, monthly_usage=50, purchase_amount=100, customer_service_calls=5, region='North'
new_cust_num = scaler_churn.transform(
    pd.DataFrame([[35, 50, 100, 5]], columns=num_cols)
)
new_cust_cat = encoder_churn.transform(
    pd.DataFrame([["North"]], columns=cat_cols)
)
new_cust_features = np.hstack([new_cust_num, new_cust_cat])

# Calculate probability and apply 0.5 decision threshold
prob_churn = model_churn.predict_proba(new_cust_features)[0][1]
is_at_risk = prob_churn >= 0.5

print(f"\nChurn Probability: {prob_churn:.2%}")
print(
    f"Classification: {'At Risk (Churn=1)' if is_at_risk else 'Safe (Churn=0)'}\n"
)

--- Part 2: Model Coefficients & Feature Impact ---
age: 0.2283
monthly_usage: -0.9102
purchase_amount: -0.0649
customer_service_calls: 1.1927
region_East: 0.0211
region_North: -0.1611
region_South: -0.3026
region_West: 0.4412
Intercept: -1.6608

Churn Probability: 54.96%
Classification: At Risk (Churn=1)



Data Source: Customer Churn Prediction Dataset https://www.kaggle.com/datasets/ziya07/customer-churn-prediction-dataset